In [1]:
!nvidia-smi

Fri May 15 14:18:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected")

CUDA available: True
GPU: Tesla T4


In [3]:
!pip install -q kaggle pandas numpy matplotlib pillow scikit-learn tqdm
!pip install -q transformers==4.51.3 accelerate bitsandbytes sentence-transformers faiss-cpu
!pip install -q evaluate rouge-score nltk gradio
!pip install -q huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 39.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.1 MB/s eta 0:00:00


In [4]:
import torch, transformers, pandas as pd
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())

Torch: 2.10.0+cu128
Transformers: 4.51.3
CUDA: True


In [5]:
from google.colab import files

uploaded = files.upload()

Saving kaggle.json to kaggle.json


In [6]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!mkdir -p data
!kaggle datasets download -d simhadrisadaram/mimic-cxr-dataset -p data --unzip

!ls -lh data

Dataset URL: https://www.kaggle.com/datasets/simhadrisadaram/mimic-cxr-dataset
License(s): unknown
100% 16.5G/16.5G [16:01<00:00, 18.4MB/s]

total 227M
-rw-r--r-- 1 root root 225M May 15 14:36 mimic_cxr_aug_train.csv
-rw-r--r-- 1 root root 1.9M May 15 14:36 mimic_cxr_aug_validate.csv
drwxr-xr-x 3 root root 4.0K May 15 14:36 official_data_iccv_final


In [7]:
import pandas as pd
import ast
import os

train_df = pd.read_csv("data/mimic_cxr_aug_train.csv")
val_df = pd.read_csv("data/mimic_cxr_aug_validate.csv")

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)

Train shape: (64586, 10)
Validation shape: (500, 10)


In [8]:
print(train_df.columns.tolist())

['Unnamed: 0.1', 'Unnamed: 0', 'subject_id', 'image', 'view', 'AP', 'PA', 'Lateral', 'text', 'text_augment']


In [9]:
train_df["image_list"] = train_df["image"].apply(ast.literal_eval)
train_df["report_list"] = train_df["text"].apply(ast.literal_eval)

val_df["image_list"] = val_df["image"].apply(ast.literal_eval)
val_df["report_list"] = val_df["text"].apply(ast.literal_eval)

print(type(train_df["image_list"].iloc[0]))
print(type(train_df["report_list"].iloc[0]))

print("\nSample image path:")
print(train_df["image_list"].iloc[0][0])

print("\nSample report:")
print(train_df["report_list"].iloc[0][0][:500])

<class 'list'>
<class 'list'>

Sample image path:
files/p10/p10000032/s50414267/02aa804e-bde0afdd-112c0b34-7bc16630-4e384014.jpg

Sample report:
Findings: There is no focal consolidation, pleural effusion or pneumothorax.  Bilateral nodular opacities that most likely represent nipple shadows. The cardiomediastinal silhouette is normal.  Clips project over the left lung, potentially within the breast. The imaged upper abdomen is unremarkable. Chronic deformity of the posterior left sixth and seventh ribs are noted. Impression: No acute cardiopulmonary process.


In [10]:
BASE_IMG_DIR = "data/official_data_iccv_final"

def build_simple_dataset(df):

    df = df.copy()

    df["first_image"] = df["image_list"].apply(lambda x: x[0])
    df["first_report"] = df["report_list"].apply(lambda x: x[0])

    df["image_path"] = df["first_image"].apply(
        lambda x: os.path.join(BASE_IMG_DIR, x)
    )

    simple_df = df[
        ["subject_id", "image_path", "view", "first_report"]
    ].copy()

    simple_df["image_exists"] = simple_df["image_path"].apply(os.path.exists)

    return simple_df


simple_train = build_simple_dataset(train_df)
simple_val = build_simple_dataset(val_df)

print("Train images exist ratio:",
      simple_train["image_exists"].mean())

print("Val images exist ratio:",
      simple_val["image_exists"].mean())

Train images exist ratio: 0.7002136685969096
Val images exist ratio: 0.676


In [11]:
simple_train = simple_train[
    simple_train["image_exists"] == True
].reset_index(drop=True)

simple_val = simple_val[
    simple_val["image_exists"] == True
].reset_index(drop=True)

print("Filtered train:", simple_train.shape)
print("Filtered val:", simple_val.shape)

simple_val.head(2)

Filtered train: (45224, 5)
Filtered val: (338, 5)


,subject_id,image_path,view,first_report,image_exists
0,10003502,data/official_data_iccv_final/files/p10/p10003...,"['AP', 'LATERAL', 'LL', 'PA']",Findings: Impression: Compared to chest radio...,True
1,10013502,data/official_data_iccv_final/files/p10/p10013...,"['PA', 'LL']",Findings: Impression:,True


In [12]:
def is_valid_report(report):

    report = str(report).strip().lower()

    if len(report) < 30:
        return False

    bad_patterns = [
        "findings: impression:",
        "findings:",
        "impression:"
    ]

    if report in bad_patterns:
        return False

    return True


simple_train["valid_report"] = simple_train["first_report"].apply(is_valid_report)
simple_val["valid_report"] = simple_val["first_report"].apply(is_valid_report)

print("Train valid reports:",
      simple_train["valid_report"].mean())

print("Val valid reports:",
      simple_val["valid_report"].mean())

Train valid reports: 0.9711436405448435
Val valid reports: 0.9822485207100592


In [13]:
simple_train = simple_train[
    simple_train["valid_report"] == True
].reset_index(drop=True)

simple_val = simple_val[
    simple_val["valid_report"] == True
].reset_index(drop=True)

print("Final train shape:", simple_train.shape)
print("Final val shape:", simple_val.shape)

print("\nExample clean report:\n")
print(simple_val.iloc[0]["first_report"][:700])

Final train shape: (43919, 6)
Final val shape: (332, 6)

Example clean report:

Findings:  Impression: Compared to chest radiographs since ___, most recently ___.  Large right and moderate left pleural effusions and severe bibasilar atelectasis are unchanged.  Cardiac silhouette is obscured.  No pneumothorax.  Pulmonary edema is mild, obscured radiographically by overlying abnormalities.


In [14]:
def report_content_length(report):
    report = str(report)
    cleaned = (
        report.replace("Findings:", "")
        .replace("findings:", "")
        .replace("Impression:", "")
        .replace("impression:", "")
        .strip()
    )
    return len(cleaned)

simple_train["content_len"] = simple_train["first_report"].apply(report_content_length)
simple_val["content_len"] = simple_val["first_report"].apply(report_content_length)

simple_train = simple_train[simple_train["content_len"] >= 80].reset_index(drop=True)
simple_val = simple_val[simple_val["content_len"] >= 80].reset_index(drop=True)

print("After content length filter:")
print("Train:", simple_train.shape)
print("Val:", simple_val.shape)

print("\nExample:")
print(simple_val.iloc[0]["first_report"][:700])

After content length filter:
Train: (42445, 7)
Val: (318, 7)

Example:
Findings:  Impression: Compared to chest radiographs since ___, most recently ___.  Large right and moderate left pleural effusions and severe bibasilar atelectasis are unchanged.  Cardiac silhouette is obscured.  No pneumothorax.  Pulmonary edema is mild, obscured radiographically by overlying abnormalities.


In [ ]:
import os

PROJECT_DIR = "cxr-intelligence-system"

train_subset = simple_train.sample(n=500, random_state=42).reset_index(drop=True)
val_subset = simple_val.sample(n=100, random_state=42).reset_index(drop=True)

os.makedirs(f"{PROJECT_DIR}/processed_data", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/src", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/results", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/app", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/docs", exist_ok=True)

train_subset.to_csv(f"{PROJECT_DIR}/processed_data/train_subset.csv", index=False)
val_subset.to_csv(f"{PROJECT_DIR}/processed_data/val_subset.csv", index=False)

print("Train subset:", train_subset.shape)
print("Val subset:", val_subset.shape)

print("\nSaved files:")
!ls -lh cxr-intelligence-system/processed_data

In [16]:
requirements_text = """
pandas
numpy
matplotlib
pillow
scikit-learn
tqdm
torch
transformers==4.51.3
accelerate
bitsandbytes
sentence-transformers
faiss-cpu
evaluate
rouge-score
nltk
gradio
"""

with open(f"{PROJECT_DIR}/requirements.txt", "w") as f:
    f.write(requirements_text.strip())

print("requirements.txt saved.")

requirements.txt saved.


# Report generation

In [6]:
report_generation_code = r'''
import torch
from PIL import Image


def generate_report_medgemma(
    image_path,
    processor,
    model,
    max_new_tokens=180
):

    image = Image.open(image_path).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {
                    "type": "text",
                    "text": (
                        "Generate a concise chest X-ray radiology report. "
                        "Use exactly this format:\n"
                        "Findings: ...\n"
                        "Impression: ..."
                    )
                },
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    generated_report = processor.decode(
        outputs[0][input_len:],
        skip_special_tokens=True
    )

    return generated_report.strip()
'''

with open(f"{PROJECT_DIR}/src/report_generation.py", "w") as f:
    f.write(report_generation_code)

print("report_generation.py updated correctly.")

report_generation.py updated correctly.


In [18]:
from huggingface_hub import login

login()

In [4]:
from transformers import AutoProcessor, AutoModelForImageTextToText

model_id = "google/medgemma-4b-it"

processor = AutoProcessor.from_pretrained(model_id)

medgemma_model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

medgemma_model.eval()

print("Loaded MedGemma.")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded MedGemma.


In [5]:
from PIL import Image

sample = val_subset.iloc[0]
image = Image.open(sample["image_path"]).convert("RGB")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "Describe this chest X-ray briefly."}
        ],
    }
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(medgemma_model.device)

input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    outputs = medgemma_model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False
    )

generated = processor.decode(
    outputs[0][input_len:],
    skip_special_tokens=True
)

print("Generated:")
print(generated)

Generated:
This chest X-ray shows a normal heart size and mediastinal structures. The lungs appear clear with no obvious consolidation or masses. The ribs and clavicles are intact. The overall image suggests a healthy chest.



In [7]:
sys.path.append(PROJECT_DIR)

from src.report_generation import generate_report_medgemma

sample = val_subset.iloc[0]

pred = generate_report_medgemma(
    image_path=sample["image_path"],
    processor=processor,
    model=medgemma_model,
    max_new_tokens=120
)

print("Ground Truth:")
print(sample["first_report"])

print("\nGenerated:")
print(pred)

Ground Truth:
Findings: AP and lateral views of the chest.  The lungs are clear.  Cardiomediastinal silhouette is within normal limits.  No acute osseous abnormality detected. Impression: No acute cardiopulmonary process.

Generated:
Findings: The heart size is normal. The mediastinum is unremarkable. The lungs are clear without evidence of consolidation, pleural effusion, or pneumothorax. The visualized bony structures are intact.
Impression: No acute cardiopulmonary abnormalities identified.


In [16]:
from tqdm import tqdm
import pandas as pd
import torch

report_results = []

N_REPORTS = 20

for idx in tqdm(range(N_REPORTS)):
    row = val_subset.iloc[idx]

    try:
        pred_report = generate_report_medgemma(
            image_path=row["image_path"],
            processor=processor,
            model=medgemma_model,
            max_new_tokens=160
        )
    except Exception as e:
        pred_report = f"ERROR: {str(e)}"

    report_results.append({
        "sample_id": idx,
        "subject_id": row["subject_id"],
        "image_path": row["image_path"],
        "ground_truth_report": row["first_report"],
        "medgemma_generated_report": pred_report
    })

    torch.cuda.empty_cache()

medgemma_results_df = pd.DataFrame(report_results)

medgemma_results_df.to_csv(
    f"{PROJECT_DIR}/results/medgemma_report_generation_results.csv",
    index=False
)

print("Saved report generation results.")
medgemma_results_df.head()

100%|██████████| 20/20 [00:00<00:00, 4567.47it/s]

Saved report generation results.


,sample_id,subject_id,image_path,ground_truth_report,medgemma_generated_report
0,0,12186603,data/official_data_iccv_final/files/p12/p12186...,Findings: AP and lateral views of the chest. ...,ERROR: name 'generate_report_medgemma' is not ...
1,1,18552428,data/official_data_iccv_final/files/p18/p18552...,Findings: Bullet fragments project over the le...,ERROR: name 'generate_report_medgemma' is not ...
2,2,10767172,data/official_data_iccv_final/files/p10/p10767...,Findings: The lungs are well expanded and clea...,ERROR: name 'generate_report_medgemma' is not ...
3,3,18043502,data/official_data_iccv_final/files/p18/p18043...,Findings: Lungs are clear. There is no eviden...,ERROR: name 'generate_report_medgemma' is not ...
4,4,10295064,data/official_data_iccv_final/files/p10/p10295...,Findings: PA and lateral views of the chest. ...,ERROR: name 'generate_report_medgemma' is not ...


In [9]:
import evaluate
import pandas as pd

rouge = evaluate.load("rouge")

predictions = medgemma_results_df["medgemma_generated_report"].tolist()
references = medgemma_results_df["ground_truth_report"].tolist()

rouge_scores = rouge.compute(
    predictions=predictions,
    references=references
)

report_metrics_df = pd.DataFrame([{
    "model": "MedGemma-4B",
    "task": "Report Generation",
    "num_samples": len(medgemma_results_df),
    "rouge1": rouge_scores["rouge1"],
    "rouge2": rouge_scores["rouge2"],
    "rougeL": rouge_scores["rougeL"],
    "rougeLsum": rouge_scores["rougeLsum"]
}])

report_metrics_df.to_csv(
    f"{PROJECT_DIR}/results/report_generation_metrics.csv",
    index=False
)

report_metrics_df

,model,task,num_samples,rouge1,rouge2,rougeL,rougeLsum
0,MedGemma-4B,Report Generation,20,0.376607,0.140324,0.28736,0.295076


# QA Dataset Creation

In [10]:
qa_generation_code = r'''
import re
import pandas as pd


def extract_findings(report):

    report = str(report)

    match = re.search(
        r"Findings:(.*?)(Impression:|$)",
        report,
        re.IGNORECASE | re.DOTALL
    )

    if match:
        return match.group(1).strip()

    return ""


def extract_impression(report):

    report = str(report)

    match = re.search(
        r"Impression:(.*)",
        report,
        re.IGNORECASE | re.DOTALL
    )

    if match:
        return match.group(1).strip()

    return ""


def yes_no_from_report(report, keywords):

    text = str(report).lower()

    if any(k.lower() in text for k in keywords):

        neg_patterns = []

        for k in keywords:
            neg_patterns.extend([
                rf"no[^.]*{re.escape(k.lower())}",
                rf"without[^.]*{re.escape(k.lower())}",
                rf"negative for[^.]*{re.escape(k.lower())}"
            ])

        for pattern in neg_patterns:
            if re.search(pattern, text):
                return "No"

        return "Yes"

    return "Not mentioned"


def create_qa_pairs(df, max_samples=100):

    qa_rows = []

    subset = df.head(max_samples).reset_index(drop=True)

    for idx, row in subset.iterrows():

        report = row["first_report"]

        findings = extract_findings(report)
        impression = extract_impression(report)

        # Findings QA
        qa_rows.append({
            "sample_id": idx,
            "subject_id": row["subject_id"],
            "image_path": row["image_path"],
            "question": "What are the findings in this chest X-ray?",
            "answer": findings if findings else report,
            "qa_type": "findings"
        })

        # Impression QA
        qa_rows.append({
            "sample_id": idx,
            "subject_id": row["subject_id"],
            "image_path": row["image_path"],
            "question": "What is the impression of this chest X-ray?",
            "answer": impression if impression else report,
            "qa_type": "impression"
        })

        # Pleural effusion
        qa_rows.append({
            "sample_id": idx,
            "subject_id": row["subject_id"],
            "image_path": row["image_path"],
            "question": "Is there pleural effusion?",
            "answer": yes_no_from_report(
                report,
                ["pleural effusion", "effusion"]
            ),
            "qa_type": "yes_no"
        })

        # Pneumothorax
        qa_rows.append({
            "sample_id": idx,
            "subject_id": row["subject_id"],
            "image_path": row["image_path"],
            "question": "Is there pneumothorax?",
            "answer": yes_no_from_report(
                report,
                ["pneumothorax"]
            ),
            "qa_type": "yes_no"
        })

        # Consolidation
        qa_rows.append({
            "sample_id": idx,
            "subject_id": row["subject_id"],
            "image_path": row["image_path"],
            "question": "Is there lung consolidation?",
            "answer": yes_no_from_report(
                report,
                ["consolidation"]
            ),
            "qa_type": "yes_no"
        })

    return pd.DataFrame(qa_rows)
'''

with open(f"{PROJECT_DIR}/src/qa_generation.py", "w") as f:
    f.write(qa_generation_code)

print("qa_generation.py saved.")

qa_generation.py saved.


In [11]:
import sys
sys.path.append(PROJECT_DIR)

from src.qa_generation import create_qa_pairs

qa_df = create_qa_pairs(
    val_subset,
    max_samples=100
)

print("QA dataset shape:", qa_df.shape)

qa_df.head(10)

QA dataset shape: (500, 6)


,sample_id,subject_id,image_path,question,answer,qa_type
0,0,12186603,data/official_data_iccv_final/files/p12/p12186...,What are the findings in this chest X-ray?,AP and lateral views of the chest. The lungs ...,findings
1,0,12186603,data/official_data_iccv_final/files/p12/p12186...,What is the impression of this chest X-ray?,No acute cardiopulmonary process.,impression
2,0,12186603,data/official_data_iccv_final/files/p12/p12186...,Is there pleural effusion?,Not mentioned,yes_no
3,0,12186603,data/official_data_iccv_final/files/p12/p12186...,Is there pneumothorax?,Not mentioned,yes_no
4,0,12186603,data/official_data_iccv_final/files/p12/p12186...,Is there lung consolidation?,Not mentioned,yes_no
5,1,18552428,data/official_data_iccv_final/files/p18/p18552...,What are the findings in this chest X-ray?,Bullet fragments project over the left humeral...,findings
6,1,18552428,data/official_data_iccv_final/files/p18/p18552...,What is the impression of this chest X-ray?,Mild pulmonary vascular congestion. Bullet fr...,impression
7,1,18552428,data/official_data_iccv_final/files/p18/p18552...,Is there pleural effusion?,No,yes_no
8,1,18552428,data/official_data_iccv_final/files/p18/p18552...,Is there pneumothorax?,No,yes_no
9,1,18552428,data/official_data_iccv_final/files/p18/p18552...,Is there lung consolidation?,Not mentioned,yes_no


In [12]:
qa_df.to_csv(
    f"{PROJECT_DIR}/results/qa_dataset.csv",
    index=False
)

print("Saved QA dataset.")
print(qa_df["qa_type"].value_counts())

Saved QA dataset.
qa_type
yes_no        300
findings      100
impression    100
Name: count, dtype: int64


In [13]:
for i in range(10):
    print("=" * 80)
    print("Q:", qa_df.iloc[i]["question"])
    print("A:", qa_df.iloc[i]["answer"])
    print("Type:", qa_df.iloc[i]["qa_type"])

Q: What are the findings in this chest X-ray?
A: AP and lateral views of the chest.  The lungs are clear.  Cardiomediastinal silhouette is within normal limits.  No acute osseous abnormality detected.
Type: findings
Q: What is the impression of this chest X-ray?
A: No acute cardiopulmonary process.
Type: impression
Q: Is there pleural effusion?
A: Not mentioned
Type: yes_no
Q: Is there pneumothorax?
A: Not mentioned
Type: yes_no
Q: Is there lung consolidation?
A: Not mentioned
Type: yes_no
Q: What are the findings in this chest X-ray?
A: Bullet fragments project over the left humeral head.  Heart size is normal.  An opacity in the left lung may represent atelectasis and mild pulmonary vascular congestion.  There is no osseous abnormality.  There is no pneumothorax or pleural effusion.
Type: findings
Q: What is the impression of this chest X-ray?
A: Mild pulmonary vascular congestion.  Bullet fragments are re- demonstrated.
Type: impression
Q: Is there pleural effusion?
A: No
Type: yes_

In [14]:
qa_medgemma_code = r'''
import torch


def answer_question_medgemma(
    question,
    context,
    processor,
    model,
    max_new_tokens=60
):

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": (
                        f"Context:\\n{context}\\n\\n"
                        f"Question: {question}\\n\\n"
                        "Answer the medical question concisely."
                    )
                }
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    decoded = processor.decode(
        outputs[0][input_len:],
        skip_special_tokens=True
    )

    return decoded.strip()
'''

with open(f"{PROJECT_DIR}/src/qa_medgemma.py", "w") as f:
    f.write(qa_medgemma_code)

print("qa_medgemma.py saved.")

qa_medgemma.py saved.


In [15]:
import sys
sys.path.append(PROJECT_DIR)

from src.qa_medgemma import answer_question_medgemma

qa_sample = qa_df.iloc[0]

question = qa_sample["question"]

context = val_subset.iloc[
    qa_sample["sample_id"]
]["first_report"]

ground_truth = qa_sample["answer"]

pred_answer = answer_question_medgemma(
    question=question,
    context=context,
    processor=processor,
    model=medgemma_model,
    max_new_tokens=60
)

print("Question:")
print(question)

print("\nGround Truth:")
print(ground_truth)

print("\nPredicted:")
print(pred_answer)

Question:
What are the findings in this chest X-ray?

Ground Truth:
AP and lateral views of the chest.  The lungs are clear.  Cardiomediastinal silhouette is within normal limits.  No acute osseous abnormality detected.

Predicted:
The lungs are clear, the cardiomediastinal silhouette is within normal limits, and there are no acute osseous abnormalities.


In [16]:
from tqdm import tqdm
import pandas as pd
import torch

qa_results = []

N_QA = 30

for idx in tqdm(range(N_QA)):

    row = qa_df.iloc[idx]

    question = row["question"]

    context = val_subset.iloc[
        row["sample_id"]
    ]["first_report"]

    ground_truth = row["answer"]

    try:

        pred_answer = answer_question_medgemma(
            question=question,
            context=context,
            processor=processor,
            model=medgemma_model,
            max_new_tokens=60
        )

    except Exception as e:

        pred_answer = f"ERROR: {str(e)}"

    qa_results.append({
        "question": question,
        "ground_truth_answer": ground_truth,
        "predicted_answer": pred_answer,
        "qa_type": row["qa_type"]
    })

    torch.cuda.empty_cache()

qa_results_df = pd.DataFrame(qa_results)

qa_results_df.head()

100%|██████████| 30/30 [00:40<00:00,  1.36s/it]


,question,ground_truth_answer,predicted_answer,qa_type
0,What are the findings in this chest X-ray?,AP and lateral views of the chest. The lungs ...,"The lungs are clear, the cardiomediastinal sil...",findings
1,What is the impression of this chest X-ray?,No acute cardiopulmonary process.,No acute cardiopulmonary process.,impression
2,Is there pleural effusion?,Not mentioned,No.,yes_no
3,Is there pneumothorax?,Not mentioned,No.,yes_no
4,Is there lung consolidation?,Not mentioned,No.,yes_no


In [17]:
qa_results_df.to_csv(
    f"{PROJECT_DIR}/results/medgemma_qa_gold_context_results.csv",
    index=False
)

print("Saved QA baseline results.")

Saved QA baseline results.


In [18]:
yes_no_df = qa_results_df[
    qa_results_df["ground_truth_answer"].isin(["Yes", "No"])
].copy()

yes_no_df["pred_clean"] = (
    yes_no_df["predicted_answer"]
    .str.lower()
    .str.strip()
    .str.replace(".", "", regex=False)
)

yes_no_df["gt_clean"] = (
    yes_no_df["ground_truth_answer"]
    .str.lower()
    .str.strip()
)

yes_no_accuracy = (
    yes_no_df["pred_clean"] == yes_no_df["gt_clean"]
).mean()

print("Gold Context Yes/No Accuracy:")
print(yes_no_accuracy)

Gold Context Yes/No Accuracy:
1.0


In [19]:
def normalize_answer(ans):
    ans = str(ans).lower().strip().replace(".", "")

    if ans.startswith("yes"):
        return "yes"
    elif ans.startswith("no"):
        return "no"
    elif "not mentioned" in ans or "not specified" in ans:
        return "not mentioned"
    else:
        return ans


yes_no_all = qa_results_df[
    qa_results_df["qa_type"] == "yes_no"
].copy()

yes_no_all["pred_clean"] = yes_no_all["predicted_answer"].apply(normalize_answer)
yes_no_all["gt_clean"] = yes_no_all["ground_truth_answer"].apply(normalize_answer)

three_class_accuracy = (
    yes_no_all["pred_clean"] == yes_no_all["gt_clean"]
).mean()

print("Gold Context Yes/No/Not-mentioned Accuracy:")
print(three_class_accuracy)

yes_no_all[[
    "question",
    "ground_truth_answer",
    "predicted_answer",
    "gt_clean",
    "pred_clean"
]].head(15)

Gold Context Yes/No/Not-mentioned Accuracy:
1.0


,question,ground_truth_answer,predicted_answer,gt_clean,pred_clean
2,Is there pleural effusion?,Not mentioned,No.,no,no
3,Is there pneumothorax?,Not mentioned,No.,no,no
4,Is there lung consolidation?,Not mentioned,No.,no,no
7,Is there pleural effusion?,No,No.,no,no
8,Is there pneumothorax?,No,No.,no,no
9,Is there lung consolidation?,Not mentioned,No. The opacity in the left lung is likely ate...,no,no
12,Is there pleural effusion?,Not mentioned,No.,no,no
13,Is there pneumothorax?,Not mentioned,No.,no,no
14,Is there lung consolidation?,Not mentioned,No.,no,no
17,Is there pleural effusion?,Not mentioned,No.,no,no


In [20]:
import evaluate
import pandas as pd

rouge = evaluate.load("rouge")

open_qa_df = qa_results_df[
    qa_results_df["qa_type"].isin(["findings", "impression"])
].copy()

open_rouge = rouge.compute(
    predictions=open_qa_df["predicted_answer"].tolist(),
    references=open_qa_df["ground_truth_answer"].tolist()
)

open_qa_metrics_df = pd.DataFrame([{
    "model": "MedGemma-4B",
    "task": "Gold Context Open QA",
    "num_samples": len(open_qa_df),
    "rouge1": open_rouge["rouge1"],
    "rouge2": open_rouge["rouge2"],
    "rougeL": open_rouge["rougeL"],
    "rougeLsum": open_rouge["rougeLsum"]
}])

open_qa_metrics_df

,model,task,num_samples,rouge1,rouge2,rougeL,rougeLsum
0,MedGemma-4B,Gold Context Open QA,12,0.772671,0.654864,0.729426,0.730736


In [21]:
def normalize_yes_no_notmentioned(ans):
    ans = str(ans).lower().strip().replace(".", "")

    if "not mentioned" in ans or "not specified" in ans or "not stated" in ans:
        return "not mentioned"
    elif ans.startswith("yes"):
        return "yes"
    elif ans.startswith("no"):
        return "no"
    else:
        return "other"


yes_no_all = qa_results_df[
    qa_results_df["qa_type"] == "yes_no"
].copy()

yes_no_all["pred_clean"] = yes_no_all["predicted_answer"].apply(normalize_yes_no_notmentioned)
yes_no_all["gt_clean"] = yes_no_all["ground_truth_answer"].apply(normalize_yes_no_notmentioned)

three_class_accuracy = (
    yes_no_all["pred_clean"] == yes_no_all["gt_clean"]
).mean()

print("Gold Context Yes/No/Not-mentioned Accuracy:")
print(three_class_accuracy)

yes_no_all[[
    "question",
    "ground_truth_answer",
    "predicted_answer",
    "gt_clean",
    "pred_clean"
]].head(15)

Gold Context Yes/No/Not-mentioned Accuracy:
0.2777777777777778


,question,ground_truth_answer,predicted_answer,gt_clean,pred_clean
2,Is there pleural effusion?,Not mentioned,No.,not mentioned,no
3,Is there pneumothorax?,Not mentioned,No.,not mentioned,no
4,Is there lung consolidation?,Not mentioned,No.,not mentioned,no
7,Is there pleural effusion?,No,No.,no,no
8,Is there pneumothorax?,No,No.,no,no
9,Is there lung consolidation?,Not mentioned,No. The opacity in the left lung is likely ate...,not mentioned,no
12,Is there pleural effusion?,Not mentioned,No.,not mentioned,no
13,Is there pneumothorax?,Not mentioned,No.,not mentioned,no
14,Is there lung consolidation?,Not mentioned,No.,not mentioned,no
17,Is there pleural effusion?,Not mentioned,No.,not mentioned,no


In [22]:
open_qa_metrics_df.to_csv(
    f"{PROJECT_DIR}/results/gold_context_open_qa_metrics.csv",
    index=False
)

yes_no_metrics_df = pd.DataFrame([{
    "model": "MedGemma-4B",
    "task": "Gold Context YesNo QA",
    "num_samples": len(yes_no_all),
    "accuracy": three_class_accuracy
}])

yes_no_metrics_df.to_csv(
    f"{PROJECT_DIR}/results/gold_context_yesno_metrics.csv",
    index=False
)

print("QA metrics saved.")

QA metrics saved.


# ColPali Retrieval + RAG

In [23]:
import gc
import torch

del medgemma_model

torch.cuda.empty_cache()
gc.collect()

print("MedGemma removed from memory.")

MedGemma removed from memory.


In [25]:
!pip install -q transformers==4.51.3
!pip install -q huggingface_hub==0.30.2
!pip install -q torchao==0.10.0
!pip install -q git+https://github.com/illuin-tech/colpali.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.4/481.4 kB 31.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.30.2 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.30.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 79.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 117.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 111.7 MB/s eta 0:00:00


In [26]:
import transformers
import huggingface_hub
import torchao

print("Transformers:", transformers.__version__)
print("HF Hub:", huggingface_hub.__version__)
print("TorchAO:", torchao.__version__)

Transformers: 4.51.3
HF Hub: 0.36.2
TorchAO: 0.10.0


In [1]:
import os
import sys
import torch
import pandas as pd

PROJECT_DIR = "cxr-intelligence-system"

sys.path.append(PROJECT_DIR)

val_subset = pd.read_csv(f"{PROJECT_DIR}/processed_data/val_subset.csv")
qa_df = pd.read_csv(f"{PROJECT_DIR}/results/qa_dataset.csv")

print("Val subset:", val_subset.shape)
print("QA dataset:", qa_df.shape)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

Val subset: (100, 7)
QA dataset: (500, 6)
CUDA: True
GPU: Tesla T4


In [14]:
rag_colpali_code = r'''
import math
import torch
from PIL import Image
from tqdm import tqdm


def move_batch_to_device_and_half(batch, device):
    new_batch = {}

    for k, v in batch.items():
        v = v.to(device)

        if torch.is_floating_point(v):
            v = v.half()

        new_batch[k] = v

    return new_batch


def load_colpali_model(model_id="vidore/colpali-v1.3"):
    from colpali_engine.models import ColPali, ColPaliProcessor

    model = ColPali.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="cuda:0"
    ).eval()

    processor = ColPaliProcessor.from_pretrained(model_id)

    return processor, model


def build_image_embeddings(df, processor, model, max_images=50):
    embeddings = []
    metadata = []

    subset = df.head(max_images).reset_index(drop=True)

    for idx, row in tqdm(subset.iterrows(), total=len(subset)):
        try:
            image = Image.open(row["image_path"]).convert("RGB")

            batch_images = processor.process_images([image])
            batch_images = move_batch_to_device_and_half(
                batch_images,
                model.device
            )

            with torch.no_grad():
                image_embedding = model(**batch_images)

            image_embedding = image_embedding.detach().cpu().float()

            embeddings.append(image_embedding)

            metadata.append({
                "sample_id": idx,
                "subject_id": row["subject_id"],
                "image_path": row["image_path"],
                "report": row["first_report"]
            })

        except Exception as e:
            print(f"Error at index {idx}: {e}")

    return embeddings, metadata


def retrieve_with_colpali(
    question,
    image_embeddings,
    metadata,
    processor,
    model,
    top_k=3
):
    batch_queries = processor.process_queries([question])
    batch_queries = move_batch_to_device_and_half(
        batch_queries,
        model.device
    )

    with torch.no_grad():
        query_embedding = model(**batch_queries)

    query_embedding = query_embedding.detach().cpu().float()

    scores = []

    for idx, image_embedding in enumerate(image_embeddings):
        try:
            score = processor.score_multi_vector(
                query_embedding,
                image_embedding
            )[0].item()
        except Exception:
            score = -1e9

        if math.isnan(score) or math.isinf(score):
            score = -1e9

        scores.append((idx, score))

    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    retrieved = []

    for idx, score in scores[:top_k]:
        item = metadata[idx].copy()
        item["score"] = score
        retrieved.append(item)

    return retrieved
'''

with open(f"{PROJECT_DIR}/src/rag_colpali.py", "w") as f:
    f.write(rag_colpali_code)

print("rag_colpali.py fixed.")

rag_colpali.py fixed.


In [1]:
!pip uninstall -y colpali-engine transformers huggingface_hub torchao -q
!pip install -q transformers==4.46.3 huggingface_hub==0.26.5
!pip install -q colpali-engine==0.3.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.8/447.8 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 51.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.26.5 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.26.5 which is incompatible.
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 107.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 27.6 MB/s eta 0:00:00

In [3]:
!pip install -q --force-reinstall numpy==1.26.4 pandas==2.2.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 117.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.1/510.1 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.3/349.3 kB 34.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.26.5 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you h

In [2]:
from colpali_engine.models import ColPali, ColPaliProcessor

print("ColPali imports OK")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

ColPali imports OK


In [15]:
import importlib
import src.rag_colpali

importlib.reload(src.rag_colpali)

from src.rag_colpali import build_image_embeddings, retrieve_with_colpali

print("Reloaded fixed ColPali code.")

Reloaded fixed ColPali code.


In [7]:
from src.rag_colpali import load_colpali_model

colpali_processor, colpali_model = load_colpali_model()

print("ColPali loaded ")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

ColPali loaded 


In [16]:
image_embeddings, metadata = build_image_embeddings(
    df=val_subset,
    processor=colpali_processor,
    model=colpali_model,
    max_images=50
)

print("Indexed images:", len(image_embeddings))
print("Metadata rows:", len(metadata))

100%|██████████| 50/50 [00:29<00:00,  1.71it/s]

Indexed images: 50
Metadata rows: 50


In [ ]:
from tqdm import tqdm
import pandas as pd

rag_rows = []

N_RAG = 30

for idx in tqdm(range(N_RAG)):

    row = qa_df.iloc[idx]
    question = row["question"]

    retrieved = retrieve_with_colpali(
        question=question,
        image_embeddings=image_embeddings,
        metadata=metadata,
        processor=colpali_processor,
        model=colpali_model,
        top_k=3
    )

    retrieved_context = "\n\n".join(
        [item["report"] for item in retrieved]
    )

    retrieved_sources = " | ".join(
        [str(item["image_path"]) for item in retrieved]
    )

    retrieval_scores = " | ".join(
        [str(round(item["score"], 3)) for item in retrieved]
    )

    rag_rows.append({
        "question": question,
        "ground_truth_answer": row["answer"],
        "qa_type": row["qa_type"],
        "retrieved_context": retrieved_context,
        "retrieved_sources": retrieved_sources,
        "retrieval_scores": retrieval_scores
    })

rag_retrieval_df = pd.DataFrame(rag_rows)

rag_retrieval_df.to_csv(
    f"{PROJECT_DIR}/results/colpali_retrieved_contexts.csv",
    index=False
)

print("Saved RAG retrieval contexts:")
print(rag_retrieval_df.shape)

rag_retrieval_df.head()

In [17]:
from src.rag_colpali import retrieve_with_colpali

question = "Is there pleural effusion?"

retrieved = retrieve_with_colpali(
    question=question,
    image_embeddings=image_embeddings,
    metadata=metadata,
    processor=colpali_processor,
    model=colpali_model,
    top_k=3
)

for item in retrieved:
    print("=" * 80)
    print("Score:", item["score"])
    print("Sample ID:", item["sample_id"])
    print("Image:", item["image_path"])
    print("Report:")
    print(item["report"][:700])

You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text and `<bos>` token after that. For this call, we will infer how many images each text has and add special tokens.


Score: 12.882556915283203
Sample ID: 18
Image: data/official_data_iccv_final/files/p19/p19845866/s54018390/a03600e3-6f4ea6f0-c413be5b-1dade32a-4447a53f.jpg
Report:
Findings: The lungs are clear without focal consolidation.  No pleural effusion or pneumothorax is seen. The cardiac and mediastinal silhouettes are unremarkable. Impression: No acute cardiopulmonary process.  No pneumothorax seen.
Score: 12.845185279846191
Sample ID: 33
Image: data/official_data_iccv_final/files/p17/p17978047/s52381069/10012b3a-50cadc6b-2c7edc30-cd4462d7-94cc5a75.jpg
Report:
Findings: Mild cardiomegaly and a calcified aorta are again seen.  The lungs remain hyperinflated, and central pulmonary arteries remain prominent.  Thin linear opacities at the lateral left base on the PA view are similar to prior, compatible with atelectasis or scarring.  There is no evidence for pulmonary consolidation, pulmonary edema, pleural effusion, or pneumothorax.  There are degenerative changes and dextroconvex scoliosis in t

In [25]:
def semantic_hit(question, gt_answer, retrieved_context):

    gt = str(gt_answer).lower()

    if gt in ["yes", "no", "not mentioned"]:
        keywords = {
            "pleural effusion": ["pleural effusion"],
            "pneumothorax": ["pneumothorax"],
            "consolidation": ["consolidation"]
        }

        q = question.lower()

        for disease, kws in keywords.items():
            if disease in q:
                return any(k in retrieved_context.lower() for k in kws)

        return False

    else:
        gt_words = set(gt.split())
        context_words = set(retrieved_context.lower().split())

        overlap = len(gt_words & context_words)

        return overlap >= 5


semantic_hits = []

for _, row in rag_retrieval_df.iterrows():

    hit = semantic_hit(
        row["question"],
        row["ground_truth_answer"],
        row["retrieved_context"]
    )

    semantic_hits.append(hit)

semantic_accuracy = sum(semantic_hits) / len(semantic_hits)

print("Clinical Semantic Retrieval Accuracy:")
print(semantic_accuracy)

Clinical Semantic Retrieval Accuracy:
0.8333333333333334


In [26]:
semantic_retrieval_metrics_df = pd.DataFrame([{
    "model": "ColPali",
    "task": "Clinical Semantic Retrieval",
    "num_queries": len(rag_retrieval_df),
    "clinical_semantic_accuracy": semantic_accuracy,
    "note": "This metric checks whether the retrieved context contains clinically relevant information for the question, not whether it retrieves the exact same patient/sample."
}])

semantic_retrieval_metrics_df.to_csv(
    f"{PROJECT_DIR}/results/colpali_semantic_retrieval_metrics.csv",
    index=False
)

semantic_retrieval_metrics_df

,model,task,num_queries,clinical_semantic_accuracy,note
0,ColPali,Clinical Semantic Retrieval,30,0.833333,This metric checks whether the retrieved conte...


# RAG answer generation

In [27]:
import gc
import torch

del colpali_model
torch.cuda.empty_cache()
gc.collect()

print("ColPali removed from GPU.")

ColPali removed from GPU.


In [29]:
!pip install -q transformers==4.51.3 huggingface_hub==0.30.2

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.30.2 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.30.2 which is incompatible.


In [1]:
import os, sys, torch, pandas as pd

PROJECT_DIR = "cxr-intelligence-system"
sys.path.append(PROJECT_DIR)

qa_df = pd.read_csv(f"{PROJECT_DIR}/results/qa_dataset.csv")
rag_retrieval_df = pd.read_csv(f"{PROJECT_DIR}/results/colpali_retrieved_contexts.csv")

print("QA:", qa_df.shape)
print("RAG retrieval:", rag_retrieval_df.shape)
print("CUDA:", torch.cuda.is_available())

QA: (500, 6)
RAG retrieval: (30, 8)
CUDA: True


In [2]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

model_id = "google/medgemma-4b-it"

processor = AutoProcessor.from_pretrained(model_id)

medgemma_model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

medgemma_model.eval()

print("MedGemma loaded again.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

MedGemma loaded again.


In [3]:
qa_medgemma_code = r'''
import torch


def answer_question_medgemma(
    question,
    context,
    processor,
    model,
    max_new_tokens=80
):

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "You are a medical QA assistant.\n"
                        "Use only the provided retrieved context.\n"
                        "Answer concisely.\n\n"
                        f"Retrieved context:\n{context}\n\n"
                        f"Question: {question}"
                    )
                }
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    decoded = processor.decode(
        outputs[0][input_len:],
        skip_special_tokens=True
    )

    return decoded.strip()
'''

with open(f"{PROJECT_DIR}/src/qa_medgemma.py", "w") as f:
    f.write(qa_medgemma_code)

print("qa_medgemma.py ready.")

qa_medgemma.py ready.


In [4]:
from src.qa_medgemma import answer_question_medgemma

row = rag_retrieval_df.iloc[0]

pred = answer_question_medgemma(
    question=row["question"],
    context=row["retrieved_context"],
    processor=processor,
    model=medgemma_model,
    max_new_tokens=80
)

print("Question:", row["question"])
print("\nGround Truth:", row["ground_truth_answer"])
print("\nRAG Answer:", pred)

Question: What are the findings in this chest X-ray?

Ground Truth: AP and lateral views of the chest.  The lungs are clear.  Cardiomediastinal silhouette is within normal limits.  No acute osseous abnormality detected.

RAG Answer: Based on the provided reports, the findings in this chest X-ray are:

*   Slight worsening of bibasilar atelectasis.
*   COPD.
*   No acute cardiopulmonary abnormality.
*   The lungs are clear.
*   Cardiomediastinal silhouette is within normal limits.
*   No acute osseous abnormality detected.


In [5]:
from tqdm import tqdm
import pandas as pd
import torch

rag_answers = []

N_FINAL_RAG = 30

for idx in tqdm(range(N_FINAL_RAG)):

    row = rag_retrieval_df.iloc[idx]

    try:

        pred = answer_question_medgemma(
            question=row["question"],
            context=row["retrieved_context"],
            processor=processor,
            model=medgemma_model,
            max_new_tokens=80
        )

    except Exception as e:

        pred = f"ERROR: {str(e)}"

    rag_answers.append({
        "question": row["question"],
        "ground_truth_answer": row["ground_truth_answer"],
        "predicted_answer": pred,
        "qa_type": row["qa_type"]
    })

    torch.cuda.empty_cache()

rag_answers_df = pd.DataFrame(rag_answers)

rag_answers_df.to_csv(
    f"{PROJECT_DIR}/results/final_rag_qa_results.csv",
    index=False
)

rag_answers_df.head()

100%|██████████| 30/30 [01:22<00:00,  2.74s/it]


,question,ground_truth_answer,predicted_answer,qa_type
0,What are the findings in this chest X-ray?,AP and lateral views of the chest. The lungs ...,"Based on the provided reports, the findings in...",findings
1,What is the impression of this chest X-ray?,No acute cardiopulmonary process.,No acute cardiopulmonary process.,impression
2,Is there pleural effusion?,Not mentioned,No.,yes_no
3,Is there pneumothorax?,Not mentioned,No.,yes_no
4,Is there lung consolidation?,Not mentioned,No.,yes_no


In [6]:
import evaluate

rouge = evaluate.load("rouge")

open_rag_df = rag_answers_df[
    rag_answers_df["qa_type"].isin(["findings", "impression"])
].copy()

rag_rouge = rouge.compute(
    predictions=open_rag_df["predicted_answer"].tolist(),
    references=open_rag_df["ground_truth_answer"].tolist()
)

rag_metrics_df = pd.DataFrame([{
    "model": "ColPali + MedGemma",
    "task": "Final RAG QA",
    "num_samples": len(open_rag_df),
    "rouge1": rag_rouge["rouge1"],
    "rouge2": rag_rouge["rouge2"],
    "rougeL": rag_rouge["rougeL"],
    "rougeLsum": rag_rouge["rougeLsum"]
}])

rag_metrics_df

,model,task,num_samples,rouge1,rouge2,rougeL,rougeLsum
0,ColPali + MedGemma,Final RAG QA,12,0.532273,0.432048,0.496064,0.521657


# Gradio

In [8]:
!pip install -q huggingface_hub==0.30.2

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.30.2 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.30.2 which is incompatible.


In [9]:
import huggingface_hub
print(huggingface_hub.__version__)

0.30.2


In [7]:
!pip install -q gradio

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.51.3 requires huggingface-hub<1.0,>=0.30.0, but you have huggingface-hub 1.15.0 which is incompatible.
tokenizers 0.21.4 requires huggingface-hub<1.0,>=0.16.4, but you have huggingface-hub 1.15.0 which is incompatible.


In [2]:
!pip install -q transformers==4.51.3 huggingface_hub==0.30.2 gradio accelerate

In [22]:
import gradio as gr
import torch

def generate_report(image):

    if image is None:
        return "Please upload a chest X-ray image."

    image = image.convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {
                    "type": "text",
                    "text": (
                        "Generate a concise chest X-ray radiology report.\n"
                        "Use exactly this format:\n\n"
                        "Findings: ...\n"
                        "Impression: ..."
                    )
                }
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=False
        )

    result = processor.decode(
        outputs[0][input_len:],
        skip_special_tokens=True
    )

    torch.cuda.empty_cache()

    return result.strip()

def medical_qa_rag(question):

    if question is None or question.strip() == "":
        return "Please enter a medical question."

    retrieved = retrieve_with_colpali(
        question=question,
        image_embeddings=image_embeddings,
        metadata=metadata,
        processor=colpali_processor,
        model=colpali_model,
        top_k=3
    )

    retrieved_context = "\n\n".join(
        [item["report"] for item in retrieved]
    )

    answer = answer_question_medgemma(
        question=question,
        context=retrieved_context,
        processor=processor,
        model=model,
        max_new_tokens=100
    )

    scores = "\n".join(
        [
            f"Source {i+1}: score = {round(item['score'], 3)}"
            for i, item in enumerate(retrieved)
        ]
    )

    final_output = (
        f"Answer:\n{answer}\n\n"
        f"Retrieved Evidence:\n{retrieved_context[:1200]}\n\n"
        f"Retrieval Scores:\n{scores}"
    )

    torch.cuda.empty_cache()

    return final_output


css = """
footer {
    visibility: hidden;
}

.gradio-container {
    max-width: 1000px !important;
}
"""

with gr.Blocks(
    title="Multi-Modal Chest X-ray Intelligence System",
    css=css
) as demo:

    gr.Markdown("# Multi-Modal Chest X-ray Intelligence System")
    gr.Markdown(
        "Dual-mode system for chest X-ray report generation and RAG-based medical question answering."
    )

    with gr.Tabs():
        with gr.Tab("Mode 1: Report Generation"):

            gr.Markdown(
                "Upload a chest X-ray image. MedGemma generates a structured radiology report."
            )

            image_input = gr.Image(
                type="pil",
                label="Upload Chest X-ray Image"
            )

            report_btn = gr.Button("Generate Report")

            report_output = gr.Textbox(
                label="Generated Report",
                lines=10
            )

            report_btn.click(
                fn=generate_report,
                inputs=image_input,
                outputs=report_output
            )

        with gr.Tab("Mode 2: RAG Medical QA"):

            gr.Markdown(
                "Ask a medical question. ColPali retrieves relevant context, then MedGemma generates the final answer."
            )

            question_input = gr.Textbox(
                label="Medical Question",
                placeholder="Example: Is there pleural effusion?"
            )

            qa_btn = gr.Button("Get RAG Answer")

            qa_output = gr.Textbox(
                label="RAG Answer",
                lines=14
            )

            qa_btn.click(
                fn=medical_qa_rag,
                inputs=question_input,
                outputs=qa_output
            )

demo.launch(share=True)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://56ba84be14227b7963.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
